In [ ]:
from langsmith import Client

client = Client()

DATASET = "thesis_vqa_type1_detailed"
EXPERIMENT_PROJECT = "eval-vqa-type1-detailed-20260108-0722-28660d70" 

# 1) all example IDs in the dataset
examples = list(client.list_examples(dataset_name=DATASET))  # :contentReference[oaicite:2]{index=2}
all_example_ids = {str(e.id) for e in examples}

# 2) example IDs that already have (root) runs in this experiment
# list_runs supports filtering by project_name and selecting fields. :contentReference[oaicite:3]{index=3}
ran_example_ids = set()
for r in client.list_runs(
    project_name=EXPERIMENT_PROJECT,
    is_root=True,                      # avoid counting child spans
    select=["reference_example_id"],   # faster
):
    if r.reference_example_id:
        ran_example_ids.add(str(r.reference_example_id))  # field documented here :contentReference[oaicite:4]{index=4}

# 3) examples that do NOT yet have runs in this experiment
missing_ids = all_example_ids - ran_example_ids
missing_examples = [e for e in examples if str(e.id) in missing_ids]

print(f"Missing examples: {len(missing_examples)}")

# Now pass `missing_examples` into `evaluate(..., data=missing_examples)` if you want.


LangSmithAuthError: Authentication failed for /datasets. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/datasets?limit=1&name=thesis_vqa_scenario1', '{"detail":"Invalid token"}')

https://smith.langchain.com/o/fc4e0a84-47c8-49fa-bbfd-1012f40919de/datasets/8d81d8c5-7acf-465a-99d6-e37f345c7910/compare?selectedSessions=27eaa54a-1565-4416-a474-20b6b06a3f7f&baseline=undefined

In [ ]:
from langsmith import Client
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env file
client = Client()

results = client.get_experiment_results(name="experiment-vqa-scenario1-20260115-0241-b9f3f5fb", preview=True)
results

{'feedback_stats': {'agent_goal_accurac': {'n': 89,
   'avg': 0.6236426966292135,
   'stdev': 0.42941207099512685,
   'errors': 0,
   'values': {},
   'type': 'primary',
   'contains_thread_feedback': False},
  'context_relevance': {'n': 175,
   'avg': 0.9695121951219512,
   'stdev': 0.10613960478981216,
   'errors': 0,
   'values': {},
   'type': 'primary',
   'contains_thread_feedback': False},
  'disease_accuracy': {'n': 89,
   'avg': 0.9314606741573034,
   'stdev': 0.20803819097203424,
   'errors': 0,
   'values': {},
   'type': 'primary',
   'contains_thread_feedback': False},
  'faithfulness': {'n': 178,
   'avg': 0.8612438202247191,
   'stdev': 0.2075267173567408,
   'errors': 0,
   'values': {},
   'type': 'primary',
   'contains_thread_feedback': False},
  'note': {'n': 1,
   'avg': None,
   'stdev': None,
   'errors': 0,
   'values': {},
   'type': 'primary',
   'contains_thread_feedback': False},
  'trajectory_accuracy': {'n': 8,
   'avg': 0.0,
   'stdev': 0.0,
   'errors': 

In [ ]:
# 1) find the runs in that experiment that are linked to this example
runs = client.list_runs(
    project_name=EXPERIMENT_PROJECT,
    reference_example_id="0c75855a-24bb-4a9f-b6f3-aae1e7b53d6e",  # filter runs by linked example :contentReference[oaicite:2]{index=2}
    is_root=True,
    select=["id"],
)

print(f"Runs for example: {[r.id for r in runs]}")

Runs for example: [UUID('019b9ae2-3fb8-7b21-a6a7-66fb21455f39')]


https://smith.langchain.com/o/fc4e0a84-47c8-49fa-bbfd-1012f40919de/datasets/8d81d8c5-7acf-465a-99d6-e37f345c7910/compare?selectedSessions=27eaa54a-1565-4416-a474-20b6b06a3f7f

In [5]:
from langsmith import Client
import json
from dotenv import load_dotenv

load_dotenv()  # take environment variables from .env file

client = Client()

experiment_name = "experiment-vqa-scenario1-20260115-0241-b9f3f5fb"  # from experiment URL
FEEDBACK_KEY = "Faithfulness"  # the evaluator key you want to fix

# 1) Get all example<->runs in the experiment (use preview=True for speed)
try:
    results = client.get_experiment_results(name=experiment_name, preview=True)
    print(f"Results keys: {list(results.keys())}")
    
    # Check what examples_with_runs actually contains
    examples_with_runs = results.get("examples_with_runs")
    print(f"Type of examples_with_runs: {type(examples_with_runs)}")
    
    if examples_with_runs is None:
        print("examples_with_runs is None, trying alternative approach...")
        # Try to get runs directly
        runs = list(client.list_runs(
            project_name=experiment_name,
            is_root=True,
            select=["id", "reference_example_id"]
        ))
        print(f"Found {len(runs)} runs directly")
        
        # Build mapping from runs
        run_id_to_example_id = {}
        run_ids = []
        for run in runs:
            if hasattr(run, 'reference_example_id') and run.reference_example_id:
                run_id_to_example_id[str(run.id)] = str(run.reference_example_id)
                run_ids.append(str(run.id))
        
        print(f"Built mapping for {len(run_ids)} runs")
    else:
        # Handle the generator properly
        run_id_to_example_id = {}
        run_ids = []
        
        # Convert to list first to avoid generator exhaustion issues
        examples_with_runs_list = list(examples_with_runs)
        print(f"Processing {len(examples_with_runs_list)} examples with runs")
        
        for ewr in examples_with_runs_list:
            # ExampleWithRuns is a Pydantic model, access attributes directly
            if hasattr(ewr, 'id') and hasattr(ewr, 'runs'):
                example_id = str(ewr.id)
                for run in ewr.runs:
                    if hasattr(run, 'id'):
                        run_id = str(run.id)
                        run_id_to_example_id[run_id] = example_id
                        run_ids.append(run_id)
            else:
                print(f"Unexpected ewr format: {type(ewr)}")
                print(f"ewr attributes: {dir(ewr)}")
        
        print(f"Found {len(run_ids)} runs to check")

    # 2) Pull feedback for that key across all runs
    if run_ids:
        feedback_iter = client.list_feedback(run_ids=run_ids, feedback_key=[FEEDBACK_KEY])
        
        bad_feedback = []
        for fb in feedback_iter:
            # Heuristic: score==0 due to evaluator error
            if fb.score == 0:
                bad_feedback.append(fb)
                
        print(f"Found {len(bad_feedback)} bad feedback items with score=0")
        
        # Show some examples
        if bad_feedback:
            print("\nFirst few bad feedback items:")
            for i, fb in enumerate(bad_feedback[:3]):
                print(f"  {i+1}. Run ID: {fb.run_id}, Score: {fb.score}, Key: {fb.key}")
        
        bad_feedback
    else:
        print("No runs found to check feedback for")
        
except Exception as e:
    print(f"Error: {e}")
    print(f"Error type: {type(e)}")
    import traceback
    traceback.print_exc()

Results keys: ['feedback_stats', 'run_stats', 'examples_with_runs']
Type of examples_with_runs: <class 'generator'>
Processing 89 examples with runs
Found 89 runs to check
Found 0 bad feedback items with score=0


https://smith.langchain.com/o/fc4e0a84-47c8-49fa-bbfd-1012f40919de/datasets/8d81d8c5-7acf-465a-99d6-e37f345c7910/compare?selectedSessions=27eaa54a-1565-4416-a474-20b6b06a3f7f&baseline=undefined

In [7]:
from langsmith import Client
from collections import defaultdict
from datetime import datetime

client = Client()

EXPERIMENT_PROJECT_ID = "27eaa54a-1565-4416-a474-20b6b06a3f7f"
FEEDBACK_KEY = "agent_goal_accuracy"
# FEEDBACK_KEY = "agent_goal_accurac"

# 1) Get root runs in the experiment + their reference_example_id
runs = list(
    client.list_runs(
        project_id=EXPERIMENT_PROJECT_ID,
        is_root=True,
        select=["id", "reference_example_id"],
    )
)
run_ids = [str(r.id) for r in runs]
run_id_to_example_id = {str(r.id): str(r.reference_example_id) for r in runs if r.reference_example_id}

# 2) Pull all feedback for that key across those runs
feedback = list(
    client.list_feedback(
        run_ids=run_ids,
        feedback_key=[FEEDBACK_KEY],
        # feedback_source_type="model",
    )
)

# 3) Group by (run_id, key, feedback_source.type)
groups = defaultdict(list)
for fb in feedback:
    source_type = getattr(getattr(fb, "feedback_source", None), "type", None)
    groups[(str(fb.run_id), fb.key, source_type)].append(fb)

def is_error_feedback(fb) -> bool:
    comment = (fb.comment or "").lower()
    return ("error during evaluation" in comment) or ("traceback" in comment)

def ts(dt):
    return dt if isinstance(dt, datetime) else datetime.min

def safe_score(fb):
    # score can be None; treat as very low
    return fb.score if fb.score is not None else float("-inf")

def pick_keep(feedback_items):
    """
    Keep policy:
    1) Prefer non-error feedbacks
    2) Among preferred set, keep the HIGHEST score
    3) If tie on score, keep most recently created
    4) Tie-break by id (stable)
    """
    non_error = [fb for fb in feedback_items if not is_error_feedback(fb)]
    pool = non_error if non_error else feedback_items

    return max(
        pool,
        key=lambda fb: (safe_score(fb), ts(fb.created_at), str(fb.id)),
    )

def dedupe_groups(groups, dry_run=True):
    deletions = []
    for (run_id, key, source_type), items in groups.items():
        if len(items) <= 1:
            continue

        keep = pick_keep(items)
        for fb in items:
            if fb.id == keep.id:
                continue
            deletions.append((run_id, key, source_type, fb, keep))

    deletions.sort(key=lambda x: (str(x[0]), str(x[1]), str(x[2]), ts(x[3].created_at)))

    print(f"Duplicate groups: {sum(1 for _, v in groups.items() if len(v) > 1)}")
    print(f"Feedback to delete: {len(deletions)}")

    for run_id, key, source_type, fb_del, fb_keep in deletions[:10]:
        print(
            f"- run_id={run_id} key={key} src={source_type}\n"
            f"  delete id={fb_del.id} created_at={fb_del.created_at} score={fb_del.score} comment={repr(fb_del.comment)}\n"
            f"  keep   id={fb_keep.id} created_at={fb_keep.created_at} score={fb_keep.score} comment={repr(fb_keep.comment)}"
        )

    if dry_run:
        print("\nDRY RUN: not deleting anything.")
        return deletions

    for _, _, _, fb_del, _ in deletions:
        client.delete_feedback(fb_del.id)

    print("Done deleting duplicates.")
    return deletions

# usage:
deletions = dedupe_groups(groups, dry_run=True)
dedupe_groups(groups, dry_run=False)


Duplicate groups: 89
Feedback to delete: 89
- run_id=019bbe07-49cc-7253-a2da-32d0b89f7567 key=agent_goal_accuracy src=model
  delete id=019bbf75-ed7f-75e2-825a-f0855aec043e created_at=2026-01-15 03:25:31.047547 score=0.0 comment='Error: RetryError[<Future at 0x7f096c2c2ae0 state=finished raised RateLimitError>]'
  keep   id=019bbfbd-0458-7f03-963b-ac7614aaf31d created_at=2026-01-15 03:39:57.264319 score=1.0 comment='The model fully achieved the reference goal. It correctly performed a visual triage based on the provided images, used the `plant_disease_identification` tool to identify "strawberry leaf scorch" as a primary candidate, and then used the `knowledgebase_search` tool to provide detailed management advice. The response excelled by not only confirming the tool\'s top result but also by identifying a strong differential diagnosis ("Angular Leaf Spot") based on visual cues and information from the knowledge base, demonstrating superior synthesis of all available information.'
- r

[('019bbe07-49cc-7253-a2da-32d0b89f7567',
  'agent_goal_accuracy',
  'model',
  Feedback(id=UUID('019bbf75-ed7f-75e2-825a-f0855aec043e'), created_at=datetime.datetime(2026, 1, 15, 3, 25, 31, 47547), modified_at=datetime.datetime(2026, 1, 15, 4, 0, 53, 927900), run_id=UUID('019bbe07-49cc-7253-a2da-32d0b89f7567'), trace_id=UUID('019bbe07-49cc-7253-a2da-32d0b89f7567'), key='agent_goal_accuracy', score=0.0, value=None, comment='Error: RetryError[<Future at 0x7f096c2c2ae0 state=finished raised RateLimitError>]', correction=None, feedback_source=FeedbackSourceBase(type='model', metadata={'__run': {'run_id': '12efb971-af21-48d2-8295-55cd484d8307'}}, user_id='814e3d3e-d559-41af-b0b3-cf345aa3f94b', user_name=None), session_id=UUID('27eaa54a-1565-4416-a474-20b6b06a3f7f'), comparative_experiment_id=None, feedback_group_id=None, extra=None),
  Feedback(id=UUID('019bbfbd-0458-7f03-963b-ac7614aaf31d'), created_at=datetime.datetime(2026, 1, 15, 3, 39, 57, 264319), modified_at=datetime.datetime(2026, 

In [8]:
from langsmith import Client
from collections import defaultdict
from datetime import datetime

client = Client()

EXPERIMENT_PROJECT_ID = "27eaa54a-1565-4416-a474-20b6b06a3f7f"
FEEDBACK_KEY = "trajectory_accuracy"
# FEEDBACK_KEY = "agent_goal_accurac"

# 1) Get root runs in the experiment + their reference_example_id
runs = list(
    client.list_runs(
        project_id=EXPERIMENT_PROJECT_ID,
        is_root=True,
        select=["id", "reference_example_id"],
    )
)
run_ids = [str(r.id) for r in runs]
run_id_to_example_id = {str(r.id): str(r.reference_example_id) for r in runs if r.reference_example_id}

# 2) Pull all feedback for that key across those runs
feedback = list(
    client.list_feedback(
        run_ids=run_ids,
        feedback_key=[FEEDBACK_KEY],
        # feedback_source_type="model",
    )
)

# 3) Group by (run_id, key, feedback_source.type)
groups = defaultdict(list)
for fb in feedback:
    source_type = getattr(getattr(fb, "feedback_source", None), "type", None)
    groups[(str(fb.run_id), fb.key, source_type)].append(fb)

def is_error_feedback(fb) -> bool:
    comment = (fb.comment or "").lower()
    return ("error during evaluation" in comment) or ("traceback" in comment)

def ts(dt):
    return dt if isinstance(dt, datetime) else datetime.min

def safe_score(fb):
    # score can be None; treat as very low
    return fb.score if fb.score is not None else float("-inf")

def pick_keep(feedback_items):
    """
    Keep policy:
    1) Prefer non-error feedbacks
    2) Among preferred set, keep the HIGHEST score
    3) If tie on score, keep most recently created
    4) Tie-break by id (stable)
    """
    non_error = [fb for fb in feedback_items if not is_error_feedback(fb)]
    pool = non_error if non_error else feedback_items

    return max(
        pool,
        key=lambda fb: (safe_score(fb), ts(fb.created_at), str(fb.id)),
    )

def dedupe_groups(groups, dry_run=True):
    deletions = []
    for (run_id, key, source_type), items in groups.items():
        if len(items) <= 1:
            continue

        keep = pick_keep(items)
        for fb in items:
            if fb.id == keep.id:
                continue
            deletions.append((run_id, key, source_type, fb, keep))

    deletions.sort(key=lambda x: (str(x[0]), str(x[1]), str(x[2]), ts(x[3].created_at)))

    print(f"Duplicate groups: {sum(1 for _, v in groups.items() if len(v) > 1)}")
    print(f"Feedback to delete: {len(deletions)}")

    for run_id, key, source_type, fb_del, fb_keep in deletions[:10]:
        print(
            f"- run_id={run_id} key={key} src={source_type}\n"
            f"  delete id={fb_del.id} created_at={fb_del.created_at} score={fb_del.score} comment={repr(fb_del.comment)}\n"
            f"  keep   id={fb_keep.id} created_at={fb_keep.created_at} score={fb_keep.score} comment={repr(fb_keep.comment)}"
        )

    if dry_run:
        print("\nDRY RUN: not deleting anything.")
        return deletions

    for _, _, _, fb_del, _ in deletions:
        client.delete_feedback(fb_del.id)

    print("Done deleting duplicates.")
    return deletions

# usage:
deletions = dedupe_groups(groups, dry_run=True)
dedupe_groups(groups, dry_run=False)


Duplicate groups: 8
Feedback to delete: 13
- run_id=019bbe21-8060-7922-b1c1-d089e8f0c792 key=trajectory_accuracy src=model
  delete id=019bbf3d-0872-76a2-9fa1-244a09d97f74 created_at=2026-01-15 01:20:10.510762 score=0.0 comment="Error: Error calling model 'gemini-3-pro-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 0', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violati

[('019bbe21-8060-7922-b1c1-d089e8f0c792',
  'trajectory_accuracy',
  'model',
  Feedback(id=UUID('019bbf3d-0872-76a2-9fa1-244a09d97f74'), created_at=datetime.datetime(2026, 1, 15, 1, 20, 10, 510762), modified_at=datetime.datetime(2026, 1, 15, 3, 42, 2, 409163), run_id=UUID('019bbe21-8060-7922-b1c1-d089e8f0c792'), trace_id=UUID('019bbe21-8060-7922-b1c1-d089e8f0c792'), key='trajectory_accuracy', score=0.0, value=None, comment="Error: Error calling model 'gemini-3-pro-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_requests_per_model_per_day, limit: 0', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc

In [9]:
from langsmith import Client
from collections import defaultdict
from datetime import datetime

client = Client()

EXPERIMENT_PROJECT_ID = "27eaa54a-1565-4416-a474-20b6b06a3f7f"
FEEDBACK_KEY = "context_relevance"
# FEEDBACK_KEY = "agent_goal_accurac"

# 1) Get root runs in the experiment + their reference_example_id
runs = list(
    client.list_runs(
        project_id=EXPERIMENT_PROJECT_ID,
        is_root=True,
        select=["id", "reference_example_id"],
    )
)
run_ids = [str(r.id) for r in runs]
run_id_to_example_id = {str(r.id): str(r.reference_example_id) for r in runs if r.reference_example_id}

# 2) Pull all feedback for that key across those runs
feedback = list(
    client.list_feedback(
        run_ids=run_ids,
        feedback_key=[FEEDBACK_KEY],
        # feedback_source_type="model",
    )
)

# 3) Group by (run_id, key, feedback_source.type)
groups = defaultdict(list)
for fb in feedback:
    source_type = getattr(getattr(fb, "feedback_source", None), "type", None)
    groups[(str(fb.run_id), fb.key, source_type)].append(fb)

def is_error_feedback(fb) -> bool:
    comment = (fb.comment or "").lower()
    return ("error during evaluation" in comment) or ("traceback" in comment)

def ts(dt):
    return dt if isinstance(dt, datetime) else datetime.min

def safe_score(fb):
    # score can be None; treat as very low
    return fb.score if fb.score is not None else float("-inf")

def pick_keep(feedback_items):
    """
    Keep policy:
    1) Prefer non-error feedbacks
    2) Among preferred set, keep the HIGHEST score
    3) If tie on score, keep most recently created
    4) Tie-break by id (stable)
    """
    non_error = [fb for fb in feedback_items if not is_error_feedback(fb)]
    pool = non_error if non_error else feedback_items

    return max(
        pool,
        key=lambda fb: (safe_score(fb), ts(fb.created_at), str(fb.id)),
    )

def dedupe_groups(groups, dry_run=True):
    deletions = []
    for (run_id, key, source_type), items in groups.items():
        if len(items) <= 1:
            continue

        keep = pick_keep(items)
        for fb in items:
            if fb.id == keep.id:
                continue
            deletions.append((run_id, key, source_type, fb, keep))

    deletions.sort(key=lambda x: (str(x[0]), str(x[1]), str(x[2]), ts(x[3].created_at)))

    print(f"Duplicate groups: {sum(1 for _, v in groups.items() if len(v) > 1)}")
    print(f"Feedback to delete: {len(deletions)}")

    for run_id, key, source_type, fb_del, fb_keep in deletions[:10]:
        print(
            f"- run_id={run_id} key={key} src={source_type}\n"
            f"  delete id={fb_del.id} created_at={fb_del.created_at} score={fb_del.score} comment={repr(fb_del.comment)}\n"
            f"  keep   id={fb_keep.id} created_at={fb_keep.created_at} score={fb_keep.score} comment={repr(fb_keep.comment)}"
        )

    if dry_run:
        print("\nDRY RUN: not deleting anything.")
        return deletions

    for _, _, _, fb_del, _ in deletions:
        client.delete_feedback(fb_del.id)

    print("Done deleting duplicates.")
    return deletions

# usage:
deletions = dedupe_groups(groups, dry_run=True)
dedupe_groups(groups, dry_run=False)


Duplicate groups: 86
Feedback to delete: 86
- run_id=019bbe07-f5a4-7e83-9a84-ec6527e4de53 key=context_relevance src=model
  delete id=019bbec5-c0e9-74c3-8ac1-7a1ee6184fd1 created_at=2026-01-14 23:09:52.668278 score=None comment=None
  keep   id=019bbf1c-f30b-7283-b519-186c6becde99 created_at=2026-01-15 00:45:12.119534 score=1.0 comment=None
- run_id=019bbe08-9edd-7b40-9fbc-e33644963fcb key=context_relevance src=model
  delete id=019bbec4-3bef-7c91-85d3-20c502403b23 created_at=2026-01-14 23:08:17.471174 score=None comment=None
  keep   id=019bbf1c-e78f-7b20-84f5-9f1ff80f2005 created_at=2026-01-15 00:45:07.157637 score=1.0 comment=None
- run_id=019bbe08-c269-7740-aa09-675096f9f0aa key=context_relevance src=model
  delete id=019bbec3-facd-77b2-9775-083813359164 created_at=2026-01-14 23:07:57.533976 score=None comment=None
  keep   id=019bbf1c-9985-7da1-ba20-45c0610973d4 created_at=2026-01-15 00:44:47.110821 score=1.0 comment=None
- run_id=019bbe09-105c-7192-b2c9-b0392cacd322 key=context_r

[('019bbe07-f5a4-7e83-9a84-ec6527e4de53',
  'context_relevance',
  'model',
  Feedback(id=UUID('019bbec5-c0e9-74c3-8ac1-7a1ee6184fd1'), created_at=datetime.datetime(2026, 1, 14, 23, 9, 52, 668278), modified_at=datetime.datetime(2026, 1, 15, 4, 0, 8, 970619), run_id=UUID('019bbe07-f5a4-7e83-9a84-ec6527e4de53'), trace_id=UUID('019bbe07-f5a4-7e83-9a84-ec6527e4de53'), key='context_relevance', score=None, value=None, comment=None, correction=None, feedback_source=FeedbackSourceBase(type='model', metadata={'__run': {'run_id': '91a2baa2-7cc3-46f6-989f-d015691812f0'}}, user_id='814e3d3e-d559-41af-b0b3-cf345aa3f94b', user_name=None), session_id=UUID('27eaa54a-1565-4416-a474-20b6b06a3f7f'), comparative_experiment_id=None, feedback_group_id=None, extra=None),
  Feedback(id=UUID('019bbf1c-f30b-7283-b519-186c6becde99'), created_at=datetime.datetime(2026, 1, 15, 0, 45, 12, 119534), modified_at=datetime.datetime(2026, 1, 15, 4, 0, 8, 970619), run_id=UUID('019bbe07-f5a4-7e83-9a84-ec6527e4de53'), trace

https://smith.langchain.com/o/fc4e0a84-47c8-49fa-bbfd-1012f40919de/datasets/8d81d8c5-7acf-465a-99d6-e37f345c7910/compare?selectedSessions=27eaa54a-1565-4416-a474-20b6b06a3f7f&baseline=undefined

In [6]:
from langsmith import Client
client = Client()

EXPERIMENT_PROJECT_ID = "27eaa54a-1565-4416-a474-20b6b06a3f7f"
FEEDBACK_KEY = "faithfulness"

# 1) Get root runs in the experiment
runs = list(
    client.list_runs(
        project_id=EXPERIMENT_PROJECT_ID,
        is_root=True,
        select=["id"],
    )
)
run_ids = [str(r.id) for r in runs]

# 2) Fetch all feedback with that key
feedback = list(
    client.list_feedback(
        run_ids=run_ids,
        feedback_key=[FEEDBACK_KEY],
    )
)

print(f"Found {len(feedback)} feedback items to delete")

# 3) Delete all of them
for fb in feedback:
    client.delete_feedback(fb.id)


Found 185 feedback items to delete


In [ ]:
# from langsmith import Client
# client = Client()

# EXPERIMENT_PROJECT_ID = "27eaa54a-1565-4416-a474-20b6b06a3f7f"
# FEEDBACK_KEY = "agent_goal_accuracy"

# # 1) Get root runs in the experiment
# runs = list(
#     client.list_runs(
#         project_id=EXPERIMENT_PROJECT_ID,
#         is_root=True,
#         select=["id"],
#     )
# )
# run_ids = [str(r.id) for r in runs]

# # 2) Fetch all feedback with that key
# feedback = list(
#     client.list_feedback(
#         run_ids=run_ids,
#         feedback_key=[FEEDBACK_KEY],
#     )
# )

# print(f"Found {len(feedback)} feedback items to delete")

# # 3) Delete all of them
# for fb in feedback:
#     client.delete_feedback(fb.id)


Found 89 feedback items to delete


In [4]:
from langsmith import Client
client = Client()

EXPERIMENT_PROJECT_ID = "27eaa54a-1565-4416-a474-20b6b06a3f7f"
FEEDBACK_KEY = "agent_goal_accurac"

# 1) Get root runs in the experiment
runs = list(
    client.list_runs(
        project_id=EXPERIMENT_PROJECT_ID,
        is_root=True,
        select=["id"],
    )
)
run_ids = [str(r.id) for r in runs]

# 2) Fetch all feedback with that key
feedback = list(
    client.list_feedback(
        run_ids=run_ids,
        feedback_key=[FEEDBACK_KEY],
    )
)

print(f"Found {len(feedback)} feedback items to delete")

# 3) Delete all of them
for fb in feedback:
    client.delete_feedback(fb.id)


Found 134 feedback items to delete
